In [ ]:
%cd ..

In [ ]:
import logging
import os

from openai import OpenAI
from dotenv import load_dotenv

from prompt_forge import Project, LLMMessage, LLMResponse
from prompt_forge.training.pipeline import TrainingConfig

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(name)s | %(message)s")

load_dotenv()

## CV Ranking with prompt-forge

This notebook demonstrates an unconventional but powerful use of `prompt-forge`:
instead of training a *system prompt* (instructions), we use the optimizer loop to
incrementally build a **ranked list of candidates** from raw CVs.

The key insight: the "prompt" is just a mutable text artifact updated each iteration.
By replacing the default meta-prompt with one that maintains a ranking, the optimizer
becomes a stateful incremental processor rather than a prompt engineer.

**Flow:**
1. Each batch = a new set of CVs fed to the optimizer
2. The optimizer updates the running ranked list
3. After all batches, the artifact **is** the output — no inference step needed

In [ ]:
# ── LLM client (swap in your own) ─────────────────────────────────────────────

def make_llm():
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

    def llm(messages: list[LLMMessage]) -> LLMResponse:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": m.role, "content": m.content} for m in messages],
            temperature=0.2,
        )
        choice = response.choices[0]
        return LLMResponse(
            text=choice.message.content,
            usage={
                "prompt_tokens": response.usage.prompt_tokens,
                "completion_tokens": response.usage.completion_tokens,
            },
        )

    return llm

llm = make_llm()

In [ ]:
# ── Hiring criteria ────────────────────────────────────────────────────────────
# This is the only thing you define as a human — what you're looking for.

HIRING_CRITERIA = """
We are hiring a Senior Machine Learning Engineer.

Ideal profile:
- Strong Python and MLOps experience
- Production ML experience (not just research/academic)
- Experience with LLMs or NLP is a strong plus
- Team leadership or mentoring experience preferred
- Startup or fast-paced environment experience is valued
""".strip()

In [ ]:
# ── Custom meta-prompt ────────────────────────────────────────────────────────
# Instead of the default "improve these instructions" meta-prompt,
# we instruct the LLM to maintain a ranked candidate list.

RANKING_META_PROMPT = """
You are a technical recruiter maintaining a ranked shortlist of candidates for a job opening.

You will receive:
1. The current ranked list (which may be empty on the first call)
2. A set of new candidate CVs to evaluate

Your job:
- Evaluate each new CV against the hiring criteria embedded in the ranked list header
- Insert each candidate into the correct position in the ranking
- For each candidate include: rank, name, a one-line fit summary, and key strengths/gaps
- Remove candidates that are clearly not a fit (below a minimum bar)
- Keep the list clean and well-structured

Return ONLY the updated ranked list. No preamble, no explanation.
Use this exact format for each entry:

## #1 — [Candidate Name]
**Fit:** [one sentence]
**Strengths:** [comma-separated]
**Gaps:** [comma-separated or "none"]
""".strip()

In [ ]:
# ── Synthetic CVs ─────────────────────────────────────────────────────────────
# In a real scenario these would be loaded from files via project.add_examples_from_directory()

CVS = [
    {
        "name": "Alice Chen",
        "cv": """Alice Chen — ML Engineer, 7 years experience.
Currently at Stripe leading the fraud detection ML platform (Python, Kubernetes, Feast).
Previously at a Series B startup, built NLP pipeline for contract analysis from scratch.
Mentors 2 junior engineers. Published 1 paper on tabular transformers.
MSc Computer Science, Stanford."""
    },
    {
        "name": "Bob Martinez",
        "cv": """Bob Martinez — Data Scientist, 4 years experience.
Works at a large bank running A/B tests and dashboards in R and Python.
No production ML deployment experience. Strong statistics background.
BSc Mathematics. No LLM or NLP experience."""
    },
    {
        "name": "Sara Kim",
        "cv": """Sara Kim — Senior ML Engineer, 6 years experience.
At a 50-person AI startup: built and shipped LLM-powered document Q&A product end-to-end.
Owns the full MLOps stack (CI/CD, model serving, monitoring). Comfortable in Python and Go.
Led a team of 3. No formal publications but active open-source contributor.
BEng Software Engineering, ETH Zurich."""
    },
    {
        "name": "David Okafor",
        "cv": """David Okafor — ML Researcher, 5 years experience (mostly academic).
PhD in NLP from CMU. Strong publications record (ACL, EMNLP).
Limited production experience — one internship at Google Brain.
Currently looking to transition from academia to industry.
Expert in transformers and fine-tuning."""
    },
    {
        "name": "Priya Nair",
        "cv": """Priya Nair — ML Platform Engineer, 8 years experience.
Built internal ML platforms at two unicorn startups (feature stores, experiment tracking, model registry).
Expert in MLflow, Ray, and Kubeflow. Mentored 5+ engineers.
Less exposure to cutting-edge model research but very strong on reliability and scale.
MSc Electrical Engineering, IIT Bombay."""
    },
    {
        "name": "Tom Reilly",
        "cv": """Tom Reilly — Junior Data Analyst, 2 years experience.
Works with SQL and Tableau at an e-commerce company. Basic Python scripting.
Recently completed an online ML course. No production ML experience.
BSc Business Administration."""
    },
    {
        "name": "Mei Wang",
        "cv": """Mei Wang — ML Engineer, 5 years experience.
At Alibaba Cloud, worked on recommendation systems at scale (hundreds of millions of users).
Strong in PyTorch and distributed training. Some LLM fine-tuning experience.
Limited English communication in past roles — remote-first setup may be a challenge.
MSc AI, Tsinghua University."""
    },
    {
        "name": "Carlos Ruiz",
        "cv": """Carlos Ruiz — ML Tech Lead, 9 years experience.
Founded and sold an AI startup (computer vision for retail). Now freelancing.
Deep Python and C++ skills. Has hired and managed teams of up to 8 engineers.
Less recent NLP/LLM exposure but extremely strong fundamentals.
PhD Computer Vision, Polytechnic University of Catalonia."""
    },
]

In [ ]:
import tempfile
from pathlib import Path

PROJECT_DIR = ".projects/cv_ranking"

project = Project(
    name="cv_ranking",
    llm=llm,
    project_dir=PROJECT_DIR,
)

project.set_bundle_schema(input=".txt")

# Seed prompt = hiring criteria header + empty ranking
# The optimizer will grow this into the actual ranked list
project.set_seed_prompt(f"""{HIRING_CRITERIA}

---
RANKED CANDIDATES
(no candidates evaluated yet)
""")

# Write each CV to a temp file — the library is file-based
cv_dir = Path(tempfile.mkdtemp(prefix="cv_ranking_"))
for entry in CVS:
    cv_path = cv_dir / f"{entry['name'].replace(' ', '_')}.txt"
    cv_path.write_text(entry["cv"], encoding="utf-8")
    project.add_example(
        bundle_id=entry["name"],
        files={"input": cv_path},
    )

print(f"Loaded {project.num_examples} CVs")

In [ ]:
# ── Run the ranking ────────────────────────────────────────────────────────────

report = project.train(
    eval_strategy=None,        # No scoring — we're building a document, not optimizing a prompt
    optimizer_kwargs={"optimizer_prompt": RANKING_META_PROMPT},
    config=TrainingConfig(
        batch_size=3,          # Process 3 CVs at a time
        max_iterations=3,      # ceil(8 CVs / 3 per batch) = 3 passes
        inference_temperature=0.1,
    ),
)

print(f"Done — {len(report.iterations)} iterations, {report.total_tokens_used:,} tokens used")

In [ ]:
# ── Final ranked list ─────────────────────────────────────────

print(project.list_versions()[-1].prompt_text)

In [ ]:
# ── Incremental view: how the ranking evolved ─────────────────────────────────

for i, iteration in enumerate(report.iterations):
    print(f"\n{'='*60}")
    print(f"After batch {i+1} ({iteration.tokens_used:,} tokens):")
    print('='*60)
    print(project.list_versions()[i+1].prompt_text)

In [ ]:
# ── Add new CVs and continue ranking ──────────────────────────────────────────
# New candidates arrive — just add them and run another training pass.
# The optimizer picks up from the latest version automatically.

new_cvs = [
    "Lena Fischer — ML Engineer, 5 years. At a Berlin AI startup, built RAG pipelines using LangChain and OpenAI APIs. Strong Python, some Rust. No formal leadership experience but very hands-on.",
    "James Wu — ML Engineer, 3 years. At Netflix recommendations team. Solid production ML experience with A/B testing and real-time serving. No NLP/LLM experience.",
]

for i, cv_text in enumerate(new_cvs):
    cv_path = cv_dir / f"new_candidate_{i+1}.txt"
    cv_path.write_text(cv_text, encoding="utf-8")
    project.add_example(
        bundle_id=f"new_candidate_{i+1}",
        files={"input": cv_path},
    )

report2 = project.train(
    train_bundles=list(project.bundles)[-len(new_cvs):],  # Only process the new ones
    eval_strategy=None,
    optimizer_kwargs={"meta_prompt": RANKING_META_PROMPT},
    config=TrainingConfig(
        batch_size=2,
        max_iterations=1,
        inference_temperature=0.1,
    ),
)

print(project.list_versions()[-1].prompt_text)